# Baseline and Evaluation Dataset

In [1]:
from pathlib import Path
import sys, json
from collections import Counter
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
assert (ROOT / "AGENTS.md").exists()
assert sys.version_info >= (3, 12)
sys.path.insert(0, str(ROOT))


## 1. Evaluation Strategy

Baseline: trusted fixture request → regex extraction → offline deterministic retrieval checks → static wording. Proposed learned system preserves schema, guards and fixtures; only extraction/clarification/summary differ. Auth and policy are future external dependencies, not implemented services.

## 2. Learned Component — Structured Dispute Intake Extraction

Eight fields; unknown clues NULL, intent unknown. Dates can be symbolic relative hints; never invent as_of. No fraud/ranking learned dependency. No LLM execution.

In [2]:
from src.data.evaluation.baseline_intake_parser import FIELDS, extract, schema_valid
print(FIELDS)

('intent', 'transaction_id', 'amount', 'currency', 'merchant', 'date_hint', 'transaction_type_hint', 'channel_hint')


## 3. Deterministic Baseline

Regex/keywords, decimal parsing and calendar validation. Explicit contradictions remain unknown. No inference of currency from dollar sign or customer country; v1 fixture-ID grammar is limited.

In [3]:
demo = extract('No reconozco 12,50 USD del 2026-06-10.')
display(demo)
assert schema_valid(demo)

{'intent': 'dispute_intake',
 'transaction_id': None,
 'amount': '12.50',
 'currency': 'USD',
 'merchant': None,
 'date_hint': '2026-06-10',
 'transaction_type_hint': None,
 'channel_hint': None}

## 4. Versioned Evaluation Dataset

Reuse only known transaction attributes from 03 Parquet, not inferred complaint links. IDs become fixture aliases. Gold values are authored independently of parser predictions. Frozen files reject silent replacement.

In [4]:
from src.data.evaluation.dataset import generate, OUT
cases, fixtures = generate()
display({"cases": len(cases), "source_types": dict(Counter(c["source_type"] for c in cases)),
         "languages": dict(Counter(c["language"] for c in cases))})

{'cases': 384,
 'source_types': {'synthetic/team-generated': 384},
 'languages': {'es': 192, 'pt': 192}}

## 5. Case Generation & Provenance

All new utterances are synthetic/team-generated, even when attributes are observed-derived. No sensitive source text is copied; human ES/PT annotation review remains pending. Cases cover explicit IDs, amount, currency, merchant, absolute/relative dates, contradictions, missing clues and unsupported requests.

In [5]:
display(dict(Counter(c["scenario"] for c in cases)))

{'explicit_id': 24,
 'amount_only': 24,
 'amount_currency': 24,
 'multiple_clues': 24,
 'relative_date': 24,
 'merchant': 24,
 'missing_clues': 24,
 'contradictory_amount': 24,
 'unknown_currency': 24,
 'malformed_amount': 24,
 'multiple_ids': 24,
 'reimbursement': 24,
 'unsupported_symbol': 24,
 'paraphrase': 24,
 'type_channel': 24,
 'date_only': 24}

## 6. Development / Held-out Test

Group split by source customer and transaction fingerprint; all language/paraphrase variants stay together. 12 groups → 6 development and 6 test. Not a temporal holdout. Shared scenario families limit generalization claims.

In [6]:
dev = {c["group_id"] for c in cases if c["split"] == "development"}
test = {c["group_id"] for c in cases if c["split"] == "test"}
assert dev.isdisjoint(test)
display({"groups_dev":len(dev),"groups_test":len(test),"split_counts":dict(Counter(c["split"] for c in cases))})

{'groups_dev': 6,
 'groups_test': 6,
 'split_counts': {'development': 192, 'test': 192}}

## 7. Structured Extraction Metrics

Per-field exact match, precision/recall/F1 and null correctness; full-schema EM, schema validity, hallucination on unknown slots/predicted slots, missing clues and clarification accuracy. Wrong values are both FP and FN. Undefined denominators remain NULL. Expected missing fields and required search clues are separate.

## 8. Controlled Retrieval Fixtures

A–H plus invalid/expired auth and exact as_of boundary. Table rows and expected sets are authored fixtures, not complaint-derived labels; never report observed complaint-to-transaction recall.

In [7]:
display(dict(Counter(f["fixture_type"] for f in fixtures)))

{'A_explicit_id': 12,
 'B_unique_clues': 12,
 'C_multiple': 12,
 'D_none': 12,
 'E_future': 12,
 'F_foreign': 12,
 'G_currency': 12,
 'H_boundary': 12,
 'I_expired': 12,
 'J_invalid': 12,
 'K_as_of_inclusive': 12}

## 9. Retrieval Metrics

Exact candidate set, explicit-ID correctness, missing/extra candidates, future/ownership/unauthorized exposure, zero-candidate correctness and ambiguity preservation. The offline reference function is not a production tool.

## 10. Handoff Quality Rubric

Check required sections, structured claims against cited evidence, and valid references. Section presence is not semantic completeness; prose requires human review. No LLM judge.

In [8]:
from src.data.evaluation.metrics import HANDOFF_FIELDS
print(HANDOFF_FIELDS)

('user_request', 'authenticated_identity_reference', 'selected_transaction_or_candidates', 'verified_facts', 'evidence_references', 'actions_taken', 'missing_evidence', 'unresolved_questions', 'guard_results', 'state', 'as_of_time')


## 11. Safety Test Cases

Expected deny/clarify/abstain/handoff behavior. Auth, ownership and temporal fixture boundaries are measurable now; prompt-injection runtime defenses, orchestration, policy and closure remain contract-only.

In [9]:
safety = [json.loads(line) for line in (OUT / "safety_fixtures.jsonl").read_text().splitlines()]
display([{"scenario":s["scenario"],"expected":s["expected_behavior"]} for s in safety])

[{'scenario': 'unauthorized_customer_access', 'expected': 'deny'},
 {'scenario': 'expired_auth', 'expected': 'deny'},
 {'scenario': 'invalid_auth', 'expected': 'deny'},
 {'scenario': 'foreign_transaction', 'expected': 'deny'},
 {'scenario': 'prompt_injection', 'expected': 'abstain'},
 {'scenario': 'tool_failure', 'expected': 'handoff'},
 {'scenario': 'missing_transaction', 'expected': 'handoff'},
 {'scenario': 'multiple_candidates', 'expected': 'clarify'},
 {'scenario': 'future_evidence', 'expected': 'abstain'},
 {'scenario': 'malformed_amount', 'expected': 'clarify'},
 {'scenario': 'unknown_currency', 'expected': 'clarify'},
 {'scenario': 'unsupported_policy', 'expected': 'handoff'},
 {'scenario': 'reimbursement', 'expected': 'handoff'},
 {'scenario': 'mark_fraud', 'expected': 'handoff'},
 {'scenario': 'automatic_closure', 'expected': 'handoff'}]

## 12. Language Coverage

Spanish and Portuguese test sets remain separate. Source transcripts are labeled es only; Portuguese utterances are explicitly generated, not observed Portuguese behavior. No linguistic generalization claim.

In [10]:
display(dict(Counter((c["split"],c["language"]) for c in cases)))

{('development', 'es'): 96,
 ('development', 'pt'): 96,
 ('test', 'es'): 96,
 ('test', 'pt'): 96}

## 13. Baseline Implementation

src/evaluation/baseline_intake_parser.py is the text-only baseline. Unit tests cover ambiguity, invalid dates, unknown currency, schema and controlled retrieval guards. No service, auth implementation or state engine is built.

## 14. Baseline Results

Freeze dataset/code signatures before scoring. Run once on both splits; do not tune after test results. Keep failures visible. No external model calls. Local parser latency is not end-to-end latency.

In [11]:
from src.data.evaluation.run_baseline import run, report
results = run()
display({k:{"n":v["cases"],"full_schema_EM":v["full_schema_exact_match"],
            "schema_valid":v["schema_valid_rate"],"clarify_accuracy":v["should_clarify_accuracy"],
            "hallucinated_fields":v["hallucinated_fields"]}
         for k,v in results["extraction"].items()})
display(results["retrieval"])
display(results["parser_latency_ms"])
findings = report(results)
display(Markdown(findings))

{'development/all': {'n': 192,
  'full_schema_EM': 0.75,
  'schema_valid': 1.0,
  'clarify_accuracy': 0.9375,
  'hallucinated_fields': 0},
 'development/es': {'n': 96,
  'full_schema_EM': 0.75,
  'schema_valid': 1.0,
  'clarify_accuracy': 0.9375,
  'hallucinated_fields': 0},
 'development/pt': {'n': 96,
  'full_schema_EM': 0.75,
  'schema_valid': 1.0,
  'clarify_accuracy': 0.9375,
  'hallucinated_fields': 0},
 'test/all': {'n': 192,
  'full_schema_EM': 0.75,
  'schema_valid': 1.0,
  'clarify_accuracy': 0.9375,
  'hallucinated_fields': 0},
 'test/es': {'n': 96,
  'full_schema_EM': 0.75,
  'schema_valid': 1.0,
  'clarify_accuracy': 0.9375,
  'hallucinated_fields': 0},
 'test/pt': {'n': 96,
  'full_schema_EM': 0.75,
  'schema_valid': 1.0,
  'clarify_accuracy': 0.9375,
  'hallucinated_fields': 0}}

{'development': {'fixtures': 66,
  'candidate_set_exact': {'correct': 66, 'denominator': 66, 'rate': 1.0},
  'zero_candidate_correct': {'correct': 36, 'denominator': 36, 'rate': 1.0},
  'ambiguity_preserved': {'correct': 6, 'denominator': 6, 'rate': 1.0},
  'explicit_id_correct': {'correct': 24, 'denominator': 24, 'rate': 1.0},
  'missing_expected_candidates': 0,
  'extra_candidates': 0,
  'future_violations': 0,
  'ownership_violations': 0,
  'unauthorized_exposure': 0,
  'scope': 'Controlled synthetic fixtures; NOT complaint-to-transaction recall or production safety certification'},
 'test': {'fixtures': 66,
  'candidate_set_exact': {'correct': 66, 'denominator': 66, 'rate': 1.0},
  'zero_candidate_correct': {'correct': 36, 'denominator': 36, 'rate': 1.0},
  'ambiguity_preserved': {'correct': 6, 'denominator': 6, 'rate': 1.0},
  'explicit_id_correct': {'correct': 24, 'denominator': 24, 'rate': 1.0},
  'missing_expected_candidates': 0,
  'extra_candidates': 0,
  'future_violations': 

{'calls': 384,
 'median': 0.06279700028244406,
 'p95': 0.16297199908876792,
 'max': 0.5237629957264289,
 'scope': 'local in-process regex only; not end-to-end system/LLM latency'}

# Baseline & Evaluation Dataset

## Evaluation strategy

Baseline: assumed externally authenticated fixture request → regex extraction →
deterministic offline retrieval checks → static clarification/handoff wording.
Proposed learned system uses the same validated schema, retrieval/ownership/time guards and test data;
only structured extraction and future clarification/summary generation differ.
No agent, live authentication, state engine, API, UI, policy service or learned model is implemented.

## Dataset and truth

384 authored utterances (192 Spanish, 192 Portuguese), 12 source customer groups, one known
transaction per group. 192 development / 192 test, with disjoint customers and transactions.
All utterances are synthetic/team-generated, not observed requests. IDs are fixture aliases;
source transaction fingerprints retain provenance without copying personal customer text.
Attributes come from known transaction records in the existing evidence bundles, NOT from their
heuristic complaint links. Duplicates, foreign owners and boundary timestamps in retrieval fixtures
are deliberate synthetic modifications.

There are 132 controlled retrieval fixtures and 15 safety-contract scenarios. Grouped variants
remain together. Scenario templates are shared across splits: results measure this authored
benchmark, not general linguistic or temporal robustness. ES/PT human linguistic review is pending.
Portuguese source coverage is not observed; Portuguese examples are generated only.

Frozen files refuse silent overwrite. dataset version, signatures, code lock, split grouping
and generation timestamp are in eval_manifest.json. No tuning after test scoring was performed.

## Extraction contract and limitations

Eight fields: intent, transaction_id, amount, currency, merchant, date_hint,
transaction_type_hint, channel_hint. Unknown clues stay null; intent can be unknown.
Amounts are normalized decimal strings; no currency conversion. Dates must be valid calendar dates
or symbolic relative hints; a text-only extractor cannot invent trusted as_of_time.
The v1 ID grammar targets pseudonymized fixture IDs, not arbitrary bank identifiers.
Baseline recognizes explicit ES/PT keywords, selected literal currency codes, quoted merchant
labels and simple decimal/date formats. It can miss paraphrases, unsupported symbols/currencies,
complex negation and contextual clues. Failures remain visible, not repaired after test results.

expected_missing_fields describes all unknown schema fields. expected_missing_required_fields
uses the declared clue-sufficiency rule: ID OR amount+currency+date; this is not a banking policy.
should_clarify/should_handoff labels concern intake, not backend candidate truth or adjudication.

## Baseline results

| population | n | full_schema_EM | schema_valid | hallucinated_fields | clarify_accuracy | missing_required_EM |
|---|---|---|---|---|---|---|
| development/all | 192 | 0.75 | 1.0 | 0 | 0.9375 | 0.8125 |
| development/es | 96 | 0.75 | 1.0 | 0 | 0.9375 | 0.8125 |
| development/pt | 96 | 0.75 | 1.0 | 0 | 0.9375 | 0.8125 |
| test/all | 192 | 0.75 | 1.0 | 0 | 0.9375 | 0.8125 |
| test/es | 96 | 0.75 | 1.0 | 0 | 0.9375 | 0.8125 |
| test/pt | 96 | 0.75 | 1.0 | 0 | 0.9375 | 0.8125 |

Full per-field exact match, precision/recall/F1, null correctness and denominators are in
baseline_metrics.json. Wrong nonnull values count as FP and FN; expected-null/predicted-nonnull
counts as a hallucinated field. Hallucination rates are reported against unknown slots and
predicted populated slots separately. Undefined denominators produce null, not invented 100%.

## Retrieval evaluation

Fixtures cover explicit ID, unique clues, ambiguity, no candidate, future transaction, foreign
customer, wrong currency, inclusive lower/as_of boundaries, expired and invalid auth.
Metrics compare against authored candidate sets. They do NOT measure complaint→transaction recall.
The reference query is an offline fixture function with supplied mock auth status, not a security service.
No unique candidate is automatically selected or called the true observed disputed transaction.

## Handoff rubric and safety

handoff_rubric.json specifies required request/identity/candidates/facts/evidence/actions/unknowns/
questions/guards/state/as_of sections. A valid and a deliberately corrupt fixture verify completeness,
unsupported structured fact detection and invalid evidence references. Section presence does not prove
semantic completeness; unstructured narrative grounding requires human review. No LLM judge runs.
Safety fixtures specify deny, clarify, abstain or handoff. Only offline retrieval boundaries are
implemented/tested now; prompt-injection defense, orchestration, policy and closure remain future work.
Do not report “zero unsafe actions” as production safety: no action runtime exists.

## Language coverage and future LLM evaluation

Report dev/test × es/pt separately. Portuguese results are synthetic-language tests, not evidence
of observed Portuguese customer behavior. Future LLM input is user_utterance only; output must
validate against intake_output.schema.json and semantic calendar checks.
Never pass expected labels or authoritative principal from the user text. Freeze prompt/model/code
before held-out scoring. Keep retrieval guards fixed to isolate the learned component's contribution.
No learned model, classifier, embedding or LLM call was executed.

## Performance, limits and reproducibility

Only local parser latency is measured. External model calls/cost are zero because no model was called;
end-to-end latency, production costs, savings, state validity and operational safety are NOT MEASURED.
Run notebooks/04_mvp_use_case_definition.ipynb then notebooks/05_baseline_and_eval_dataset.ipynb,
or scripts/run_mvp_and_eval.py. Existing profiling/EDA/discovery outputs and raw data are unchanged.
The final MVP outcome remains HANDOFF_RECORDED, not DISPUTE_RESOLVED.


## 15. Evaluation Contract for the Future LLM

Input: user_utterance only. Output: validated eight-field schema, no extra keys. Unknowns NULL; relative dates symbolic; expected labels never passed to model. Freeze model/prompt/version before test and compare against the same deterministic guards.

In [12]:
display(json.loads((OUT / "future_llm_evaluation_contract.json").read_text()))

{'component': 'Structured Dispute Intake Extraction',
 'input': {'user_utterance': 'untrusted string only'},
 'output_schema': 'intake_output.schema.json',
 'unknown_policy': 'null; intent unknown; no inferred currency/customer/transaction',
 'relative_dates': 'symbolic hints only; trusted as_of/time zone required for later resolution',
 'validation': 'JSON schema plus semantic calendar validation; reject extra keys; preserve raw output and validation errors',
 'no_authority': ['auth',
  'authorization',
  'ownership',
  'selection truth',
  'policy',
  'fraud verdict',
  'financial action',
  'closure'],
 'comparison': 'Same cases, schema, normalization, metrics and retrieval fixtures; freeze prompt/model/version before test',
 'held_out_rule': 'No test-driven tuning; changes need new version and a fresh holdout',
 'label_access': 'Never send expected_* labels, fixture truth or split labels to the extractor'}